In [64]:
import numpy as np
import pandas as pd


In [65]:
# Load in movies dataset from parent directory

# movies = pd.read_csv('../ml-32m/movies.csv')
# ratings = pd.read_csv('../ml-32m/ratings.csv')
# movies.head()

# Code for 100k dataset:
ratings = pd.read_csv(
    "../ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

movies = pd.read_csv(
    "../ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None
)

# Name movies columns
movies = movies[[0, 1]]
movies.columns = ["movie_id", "title"]

print(movies.head())

   movie_id              title
0         1   Toy Story (1995)
1         2   GoldenEye (1995)
2         3  Four Rooms (1995)
3         4  Get Shorty (1995)
4         5     Copycat (1995)


In [66]:
ratings.head()


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [67]:
# Use this if files are uploaded on google drive:

'''
from google.colab import drive
drive.mount('/content/drive')

movies = pd.read_csv('../ml-32m/movies.csv')
ratings = pd.read_csv('../ml-32m/ratings.csv')
'''

"\nfrom google.colab import drive\ndrive.mount('/content/drive')\n\nmovies = pd.read_csv('../ml-32m/movies.csv')\nratings = pd.read_csv('../ml-32m/ratings.csv')\n"

In [68]:
# Ratings dataset is too big! Reduce to 400,000 rows (~2500 users)

ratings = ratings.sample(n=10000, random_state=42)

In [69]:
# Merge datasets on movie ID

df = ratings.merge(movies, on='movie_id')


# Create user-movie matrix

user_movie_matrix = df.pivot_table(
    index="user_id",
    columns="title",
    values="rating"
)

In [70]:
# Simple recommender function

def recommend_movies(movie_title, top_movies=10):

    # Get ratings for selected movie
    movie_ratings = user_movie_matrix[movie_title]

    # Find correlation between this movie and other
    similar_movies = user_movie_matrix.corrwith(movie_ratings)

    # Convert correlations to dataframe
    corr_df = pd.DataFrame(similar_movies, columns=["correlation"])

    # Sort correlations highest to lowest
    recommendations = corr_df.sort_values(by='correlation', ascending=False)

    # Remove the movie itself
    recommendations = recommendations.drop(movie_title, errors="ignore")

    # Return top movies
    return recommendations.head(top_movies)


In [71]:
# recommend_movies('Love Actually (2003)')

In [72]:
# Slightly more complicated recommender function

def recommend_movies(movie_title, top_movies=10):

    # Get ratings for selected movie
    movie_ratings = user_movie_matrix[movie_title]

    # Find correlation between this movie and other
    similar_movies = user_movie_matrix.corrwith(movie_ratings)

    # Convert correlations to dataframe
    corr_df = pd.DataFrame(similar_movies, columns=["correlation"])

    # Remove NaN values
    corr_df = corr_df.dropna()

    # Count number of ratings per movie
    rating_counts = df.groupby("title")["rating"].count()

    # Add rating counts
    corr_df["num_ratings"] = rating_counts

    # Filter out unpopular movies
    recommendations = corr_df[corr_df["num_ratings"] >= 30].sort_values(
        by="correlation",
        ascending=False)

    # Remove the movie itself
    recommendations = recommendations.drop(movie_title, errors="ignore")

    # Return top movies
    return recommendations.head(top_movies)


In [73]:
# recommend_movies("Rosemary's Baby (1968)")

In [74]:
# movie predictor

movie_corrs_df = user_movie_matrix.corr()

In [75]:
movie_corrs_df = movie_corrs_df.dropna()
movie_corrs_df.head()

title,1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",8 1/2 (1963),...,Wonderland (1997),"World of Apu, The (Apur Sansar) (1959)","Wrong Trousers, The (1993)",Wyatt Earp (1994),Year of the Horse (1997),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997)
title,,,,,,,,,,,,,,,,,,,,,


In [76]:
ratings = pd.read_csv(
    "../ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

In [77]:
ratings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
